<a href="https://colab.research.google.com/github/yakshith028/AI-Assisted-Threat-Detection-Dashboard-/blob/main/sqlinfo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sqlite3
import pandas as pd

# Create in-memory database
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

# Create login logs table
cursor.execute("""
CREATE TABLE login_logs (
    id INTEGER PRIMARY KEY,
    username TEXT,
    failed_logins INTEGER,
    login_time DATE,
    country TEXT
)
""")

# Insert sample records
logs = [
    ("Alice",1,"2026-01-10","USA"),
    ("Bob",5,"2026-01-11","Canada"),
    ("Alice",2,"2026-01-15","USA"),
    ("Charlie",0,"2026-02-01","India"),
    ("Bob",7,"2026-02-10","Canada"),
    ("David",1,"2026-02-18","Germany"),
    ("Alice",8,"2026-03-02","USA"),
    ("Eve",6,"2026-03-12","India"),
    ("Frank",3,"2026-03-20","Australia"),
    ("Grace",9,"2026-03-25","USA")
]

cursor.executemany("""
INSERT INTO login_logs(username,failed_logins,login_time,country)
VALUES (?,?,?,?)
""", logs)

conn.commit()

In [9]:
#1. Display Complete Login Logs
query = """
SELECT *
FROM login_logs;
"""

print("Complete Login Logs")
print(pd.read_sql_query(query, conn))

Complete Login Logs
   id username  failed_logins  login_time    country
0   1    Alice              1  2026-01-10        USA
1   2      Bob              5  2026-01-11     Canada
2   3    Alice              2  2026-01-15        USA
3   4  Charlie              0  2026-02-01      India
4   5      Bob              7  2026-02-10     Canada
5   6    David              1  2026-02-18    Germany
6   7    Alice              8  2026-03-02        USA
7   8      Eve              6  2026-03-12      India
8   9    Frank              3  2026-03-20  Australia
9  10    Grace              9  2026-03-25        USA


In [2]:
#2. Display Complete Login Logs
query = """
SELECT id,
       username,
       failed_logins,
       country
FROM login_logs
WHERE failed_logins >= 5
ORDER BY failed_logins DESC;
"""

print("\nHigh Risk Accounts")
print(pd.read_sql_query(query, conn))


High Risk Accounts
   id username  failed_logins country
0  10    Grace              9     USA
1   7    Alice              8     USA
2   5      Bob              7  Canada
3   8      Eve              6   India
4   2      Bob              5  Canada


In [3]:
#3. Detect High-Risk Login Attempts
query = """
SELECT
    username,
    failed_logins,
    CASE
        WHEN failed_logins >= 8 THEN 'Critical'
        WHEN failed_logins >= 5 THEN 'High'
        WHEN failed_logins >= 2 THEN 'Medium'
        ELSE 'Low'
    END AS threat_level
FROM login_logs;
"""

print("\nThreat Classification")
print(pd.read_sql_query(query, conn))


Threat Classification
  username  failed_logins threat_level
0    Alice              1          Low
1      Bob              5         High
2    Alice              2       Medium
3  Charlie              0          Low
4      Bob              7         High
5    David              1          Low
6    Alice              8     Critical
7      Eve              6         High
8    Frank              3       Medium
9    Grace              9     Critical


In [4]:
#4. Identify Repeat Attackers (GROUP BY + HAVING)
query = """
SELECT
    username,
    COUNT(*) AS total_attempts,
    SUM(failed_logins) AS total_failed_logins,
    ROUND(AVG(failed_logins),2) AS average_failed_logins
FROM login_logs
GROUP BY username
HAVING COUNT(*) >= 2;
"""

print("\nRepeat Attackers")
print(pd.read_sql_query(query, conn))


Repeat Attackers
  username  total_attempts  total_failed_logins  average_failed_logins
0    Alice               3                   11                   3.67
1      Bob               2                   12                   6.00


In [5]:
#5. Region-Based Threat Analysis
query = """
SELECT
    username,
    country,
    CASE
        WHEN country IN ('USA','Canada') THEN 'North America'
        WHEN country='India' THEN 'Asia'
        WHEN country='Germany' THEN 'Europe'
        ELSE 'Other'
    END AS region
FROM login_logs;
"""

print("\nRegion Analysis")
print(pd.read_sql_query(query, conn))


Region Analysis
  username    country         region
0    Alice        USA  North America
1      Bob     Canada  North America
2    Alice        USA  North America
3  Charlie      India           Asia
4      Bob     Canada  North America
5    David    Germany         Europe
6    Alice        USA  North America
7      Eve      India           Asia
8    Frank  Australia          Other
9    Grace        USA  North America


In [6]:
#6. Monthly Attack Trend
query = """
SELECT
    strftime('%m',login_time) AS month,
    COUNT(*) AS total_logins,
    SUM(failed_logins) AS failed_attempts
FROM login_logs
GROUP BY month
ORDER BY month;
"""

print("\nMonthly Attack Trend")
print(pd.read_sql_query(query, conn))


Monthly Attack Trend
  month  total_logins  failed_attempts
0    01             3                8
1    02             3                8
2    03             4               26


In [7]:
#7. Search Specific Users (LIKE)
query = """
SELECT *
FROM login_logs
WHERE username LIKE 'A%';
"""

print("\nUsers Starting With A")
print(pd.read_sql_query(query, conn))


Users Starting With A
   id username  failed_logins  login_time country
0   1    Alice              1  2026-01-10     USA
1   3    Alice              2  2026-01-15     USA
2   7    Alice              8  2026-03-02     USA


In [8]:
#8. Top Attackers
query = """
SELECT
    username,
    failed_logins
FROM login_logs
ORDER BY failed_logins DESC
LIMIT 5;
"""

print("\nTop Attackers")
print(pd.read_sql_query(query, conn))


Top Attackers
  username  failed_logins
0    Grace              9
1    Alice              8
2      Bob              7
3      Eve              6
4      Bob              5
